# 00c. Pandas 80/20 Mastery Guide (Clinical Tables, Genomics & Safe Splits)

Before your PyTorch model sees a single number, **Pandas (`pd`)** loads, cleans, joins, and splits your dataset.  
In biomedical AI, **bad Pandas code = patient data leakage**, which ruins the entire paper!

### The 20% of Pandas You Use 80% of the Time:
1. Creating & inspecting a `DataFrame` (`.head()`, `.shape`, `.dtypes`, `.describe()`)
2. Selecting columns & rows (`.loc` vs `.iloc`)
3. Filtering with conditions (`&`, `|`, `.isin()`)
4. Handling missing values (`isna`, `fillna`, `dropna`)
5. Encoding labels & transforming columns (`.map()`, `.apply()`)
6. `value_counts()` and **`groupby().agg()`** (crucial for patient-level analysis!)
7. Merging two tables (`pd.merge`)
8. **Patient-safe Train/Test split** + converting to NumPy (`.to_numpy()`)

## 1. Series vs. DataFrame & Basic Inspection

* **`DataFrame`:** A 2D labeled table (like Excel, but programmable).
* **`Series`:** A single column extracted from a `DataFrame` (`df["diagnosis"]`).
* Always start any dataset exploration with `.shape`, `.dtypes`, `.head()`, and `.describe()`.

In [ ]:
import pandas as pd
import numpy as np

# Mock clinical respiratory dataset (notice multiple recordings per patient!)
df = pd.DataFrame({
    "recording_id": ["R1", "R2", "R3", "R4", "R5", "R6", "R7"],
    "patient_id":   ["P01", "P01", "P02", "P03", "P03", "P04", "P05"],
    "age":          [67, 67, 45, 72, 72, np.nan, 58],
    "fev1":         [0.44, 0.46, 0.91, 0.38, 0.40, 0.85, 0.52],
    "diagnosis":    ["COPD", "COPD", "Healthy", "COPD", "COPD", "Healthy", "COPD"],
})

print("Shape (rows, cols):", df.shape)
print("\nColumn types:\n", df.dtypes)
df.head(5)

## 2. Selecting Columns & Rows (`.loc` vs `.iloc`)

* `df["age"]` → grabs 1 column as a `Series`
* `df[["patient_id", "fev1"]]` → notice **double brackets** `[[...]]` to grab multiple columns!
* **`.loc[row_labels, col_names]`** → selects by **name/label** or boolean condition
* **`.iloc[row_positions, col_positions]`** → selects by **integer position** (just like NumPy slicing!)

In [ ]:
print("Single column:\n", df["diagnosis"].head(3))
print("\nMultiple columns:\n", df[["patient_id", "fev1"]].head(3))

# .iloc = integer positions (rows 0..2, cols 1..3)
print("\n.iloc[0:3, 1:4]:\n", df.iloc[0:3, 1:4])

# .loc = label/condition + named columns
print("\n.loc rows 0..2 for patient_id & diagnosis:\n", df.loc[0:2, ["patient_id", "diagnosis"]])

## 3. Filtering Rows with Conditions (`&`, `|`, `.isin()`)

Notice a crucial Pandas syntax rule:
* In normal Python we write `and` / `or`.
* Inside Pandas `[...]`, you **MUST** use **`&` (AND)**, **`|` (OR)**, and **`~` (NOT)**, AND wrap each condition in parentheses `(...)`!
* Always add `.copy()` if you plan to modify the filtered table later (avoids Pandas' famous `SettingWithCopyWarning`).

In [ ]:
# Filter: COPD patients with age >= 65
elderly_copd = df[(df["diagnosis"] == "COPD") & (df["age"] >= 65)].copy()
print("Elderly COPD rows:")
print(elderly_copd[["recording_id", "patient_id", "age", "fev1"]])

# .isin([...]) matches any item in a list cleanly
selected = df[df["patient_id"].isin(["P01", "P04"])]
print("\nOnly P01 and P04:", selected["recording_id"].tolist())

## 4. Handling Missing Values (`isna`, `fillna`, `dropna`)

Real clinical & TCGA datasets ALWAYS have `NaN`s (missing lab measurements). PyTorch will output `NaN` loss if even **one** `NaN` enters your network!
* `df.isna().sum()` shows missing count per column.
* `df["col"].fillna(value)` fills `NaN`s (e.g., with the median).
* `df.dropna(subset=[...])` drops rows where a critical column is missing.

In [ ]:
print("Missing values per column:\n", df.isna().sum())

# Strategy 1: Fill missing age with median age (keep a clean copy)
df_clean = df.copy()
median_age = df_clean["age"].median()
df_clean["age"] = df_clean["age"].fillna(median_age)
print("\nAfter fillna (median age =", median_age, "):")
print(df_clean[["patient_id", "age"]])

## 5. Creating Columns & Mapping String Labels → Integers (`.map()`)

Models need integer labels (`0, 1`), not strings (`"Healthy"`, `"COPD"`).  
`.map({"Healthy": 0, "COPD": 1})` converts string classes to clean integer targets in 1 line.

In [ ]:
label_map = {"Healthy": 0, "COPD": 1}
df_clean["target"] = df_clean["diagnosis"].map(label_map)
df_clean["low_fev1"] = (df_clean["fev1"] < 0.5).astype(int)

print(df_clean[["recording_id", "diagnosis", "target", "low_fev1"]])

## 6. `value_counts()` and `groupby().agg()`

* **`value_counts()`** immediately shows class imbalance (e.g., 80% COPD vs 20% Healthy).
* **`groupby("patient_id")`** splits the table into sub-tables—one per patient—and summarizes each patient! Used in every clinical dataset to check how many clips belong to each patient.

In [ ]:
print("Class balance:\n", df_clean["diagnosis"].value_counts())

patient_summary = (
    df_clean.groupby("patient_id")
    .agg(
        n_recordings=("recording_id", "count"),
        mean_fev1=("fev1", "mean"),
        diagnosis=("diagnosis", "first"),
    )
    .reset_index()
)
print("\nPatient-level summary:")
print(patient_summary)

## 7. Merging Tables (`pd.merge`)

In cancer genomics & clinical ML, **features** (e.g. RNA-seq / drug measurements) often live in one file, while **clinical survival / labels** live in another file.  
`pd.merge(left, right, on="patient_id", how="inner")` joins them accurately by matching IDs!

In [ ]:
clinical_notes = pd.DataFrame({
    "patient_id": ["P01", "P02", "P03", "P05"],
    "smoker":     ["Yes", "No", "Yes", "Yes"],
})

# Inner join keeps only patients present in BOTH tables; 'left' keeps all rows of df_clean!
merged = pd.merge(df_clean, clinical_notes, on="patient_id", how="left")
print(merged[["recording_id", "patient_id", "diagnosis", "smoker"]])

## 8. Patient-Safe Splitting & Converting Pandas → NumPy (`.to_numpy()`)

### ⚠️ Clinical ML Rule: NEVER split rows randomly if a patient has multiple recordings!
Look at `df_clean`: `P01` has `R1` and `R2`. If `R1` goes to train and `R2` goes to test, the CNN memorizes `P01`'s heartbeat/microphone noise instead of COPD!  
Instead, split by **`unique patient_id`**, filter rows with `.isin(...)`, assert zero overlap with `set()`, and export clean `np.ndarray`s with `.to_numpy()`.

In [ ]:
unique_patients = df_clean["patient_id"].unique()
train_pids = set(unique_patients[:3])   # P01, P02, P03
test_pids = set(unique_patients[3:])    # P04, P05

# Mandatory zero-leakage assertion!
assert train_pids.isdisjoint(test_pids), "LEAKAGE DETECTED!"

train_df = df_clean[df_clean["patient_id"].isin(train_pids)]
test_df = df_clean[df_clean["patient_id"].isin(test_pids)]

feature_cols = ["age", "fev1"]
X_train = train_df[feature_cols].to_numpy(dtype=np.float32)
y_train = train_df["target"].to_numpy(dtype=np.int64)
X_test = test_df[feature_cols].to_numpy(dtype=np.float32)
y_test = test_df["target"].to_numpy(dtype=np.int64)

print("X_train shape:", X_train.shape, X_train.dtype, "| y_train:", y_train.shape, y_train.dtype)
print("X_test shape:", X_test.shape, "| y_test:", y_test.shape)

---
## 🎯 Practice Challenges (Try Without Peeking!)

1. Using `df_clean`, group by `"diagnosis"` and compute the **mean and standard deviation** of `"fev1"` and `"age"`.
2. Add a column `"fev1_pct"` equal to `df_clean["fev1"] * 100`.
3. Filter `merged` to find all recordings where `smoker == "Yes"` AND `fev1 < 0.50`, then convert `["age", "fev1"]` of those rows into a `float32` NumPy array.